# License Plate Detection — Evaluation & Comparison
**CMPS 261 — Machine Learning Project**

This notebook loads saved metrics from both models, compares their performance, and generates the final comparison plots.

In [ ]:
import json
import os
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import pandas as pd
from PIL import Image
import random

RESULTS_DIR = '../results'

## 1. Load Metrics from Both Models

In [ ]:
with open(os.path.join(RESULTS_DIR, 'yolo_metrics.json')) as f:
    yolo = json.load(f)

with open(os.path.join(RESULTS_DIR, 'fasterrcnn_metrics.json')) as f:
    frcnn = json.load(f)

print('YOLOv8n metrics:')
print(json.dumps(yolo, indent=2))
print()
print('Faster R-CNN metrics:')
print(json.dumps(frcnn, indent=2))

## 2. Side-by-Side Metric Comparison Table

In [ ]:
# Build a unified comparison table
rows = []

# YOLO metrics (uses mAP)
rows.append({
    'Model'    : 'YOLOv8n',
    'Precision': yolo['precision'],
    'Recall'   : yolo['recall'],
    'mAP@0.5'  : yolo['map50'],
    'mAP@0.5:0.95': yolo['map50_95'],
})

# Faster R-CNN metrics
rows.append({
    'Model'    : 'Faster R-CNN',
    'Precision': frcnn['precision'],
    'Recall'   : frcnn['recall'],
    'mAP@0.5'  : frcnn.get('map50', frcnn.get('mean_iou', '-')),
    'mAP@0.5:0.95': frcnn.get('map50_95', '-'),
})

df = pd.DataFrame(rows).set_index('Model')
print(df.to_string())
df

## 3. Bar Chart Comparison

In [ ]:
metrics_to_plot = ['Precision', 'Recall', 'mAP@0.5']
models = df.index.tolist()
x = np.arange(len(metrics_to_plot))
width = 0.35

fig, ax = plt.subplots(figsize=(9, 5))
colors = ['#4C9BE8', '#E87B4C']

for i, model in enumerate(models):
    vals = [df.loc[model, m] for m in metrics_to_plot]
    bars = ax.bar(x + i * width, vals, width, label=model, color=colors[i], alpha=0.85)
    for bar, val in zip(bars, vals):
        if isinstance(val, float):
            ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                    f'{val:.3f}', ha='center', va='bottom', fontsize=9)

ax.set_xticks(x + width / 2)
ax.set_xticklabels(metrics_to_plot, fontsize=11)
ax.set_ylim(0, 1.1)
ax.set_ylabel('Score')
ax.set_title('Model Comparison — License Plate Detection')
ax.legend(fontsize=10)
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'model_comparison.png'), dpi=150)
plt.show()
print('Saved: results/model_comparison.png')

## 4. YOLO Training Curve (from saved results)

In [ ]:
import csv

yolo_results_csv = '../models/yolov8n/results.csv'

if os.path.exists(yolo_results_csv):
    yolo_df = pd.read_csv(yolo_results_csv)
    yolo_df.columns = yolo_df.columns.str.strip()

    fig, axes = plt.subplots(1, 2, figsize=(13, 4))

    # Loss
    axes[0].plot(yolo_df['epoch'], yolo_df['train/box_loss'], label='Train Box Loss')
    axes[0].plot(yolo_df['epoch'], yolo_df['val/box_loss'],   label='Val Box Loss')
    axes[0].set_title('YOLOv8n — Box Loss'); axes[0].set_xlabel('Epoch')
    axes[0].legend(); axes[0].grid(alpha=0.3)

    # mAP
    axes[1].plot(yolo_df['epoch'], yolo_df['metrics/mAP50(B)'],    label='mAP@0.5')
    axes[1].plot(yolo_df['epoch'], yolo_df['metrics/mAP50-95(B)'], label='mAP@0.5:0.95')
    axes[1].set_title('YOLOv8n — mAP'); axes[1].set_xlabel('Epoch')
    axes[1].legend(); axes[1].grid(alpha=0.3)

    plt.tight_layout()
    plt.savefig(os.path.join(RESULTS_DIR, 'yolo_training_curve.png'), dpi=150)
    plt.show()
    print('Saved: results/yolo_training_curve.png')
else:
    print('YOLO results CSV not found — run notebook 02 first.')

## 5. Side-by-Side Predictions (YOLO vs Faster R-CNN)

In [ ]:
import torch
import sys
sys.path.append('..')
from ultralytics import YOLO as UltralyticsYOLO
from torchvision.models.detection import fasterrcnn_resnet50_fpn_v2, FasterRCNN_ResNet50_FPN_V2_Weights
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
import torchvision.transforms.functional as TF

DEVICE = torch.device('mps' if torch.backends.mps.is_available() else 'cpu')

# Load YOLO
yolo_model = UltralyticsYOLO('../models/yolov8n/weights/best.pt')

# Load Faster R-CNN
def load_frcnn(path, device):
    weights = FasterRCNN_ResNet50_FPN_V2_Weights.DEFAULT
    m = fasterrcnn_resnet50_fpn_v2(weights=weights)
    in_features = m.roi_heads.box_predictor.cls_score.in_features
    m.roi_heads.box_predictor = FastRCNNPredictor(in_features, 2)
    m.load_state_dict(torch.load(path, map_location=device))
    m.to(device).eval()
    return m

frcnn_model = load_frcnn('../models/fasterrcnn_best.pth', DEVICE)
print('Both models loaded.')

In [ ]:
test_img_dir = '../data/yolo/images/test'
test_images  = random.sample(os.listdir(test_img_dir), 4)

fig, axes = plt.subplots(4, 3, figsize=(14, 16))
CONF = 0.4

for row_idx, fname in enumerate(test_images):
    img_path = os.path.join(test_img_dir, fname)
    img_pil  = Image.open(img_path).convert('RGB')
    img_np   = np.array(img_pil)

    # Col 0: original
    axes[row_idx][0].imshow(img_np)
    axes[row_idx][0].set_title('Original', fontsize=9)
    axes[row_idx][0].axis('off')

    # Col 1: YOLO
    yolo_result = yolo_model.predict(img_path, conf=CONF, verbose=False)[0]
    axes[row_idx][1].imshow(img_np)
    for box in yolo_result.boxes:
        x1, y1, x2, y2 = box.xyxy[0].tolist()
        conf = box.conf[0].item()
        rect = patches.Rectangle((x1, y1), x2-x1, y2-y1,
                                   linewidth=2, edgecolor='lime', facecolor='none')
        axes[row_idx][1].add_patch(rect)
        axes[row_idx][1].text(x1, y1-4, f'{conf:.2f}', color='lime', fontsize=8,
                              bbox=dict(facecolor='black', alpha=0.4, pad=1))
    axes[row_idx][1].set_title('YOLOv8n', fontsize=9)
    axes[row_idx][1].axis('off')

    # Col 2: Faster R-CNN
    img_tensor = TF.to_tensor(img_pil).unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        pred = frcnn_model(img_tensor)[0]
    axes[row_idx][2].imshow(img_np)
    for box, score in zip(pred['boxes'], pred['scores']):
        if score < CONF: continue
        x1, y1, x2, y2 = box.cpu().tolist()
        rect = patches.Rectangle((x1, y1), x2-x1, y2-y1,
                                   linewidth=2, edgecolor='#FF6B6B', facecolor='none')
        axes[row_idx][2].add_patch(rect)
        axes[row_idx][2].text(x1, y1-4, f'{score:.2f}', color='#FF6B6B', fontsize=8,
                              bbox=dict(facecolor='black', alpha=0.4, pad=1))
    axes[row_idx][2].set_title('Faster R-CNN', fontsize=9)
    axes[row_idx][2].axis('off')

plt.suptitle('Side-by-Side: Original | YOLOv8n | Faster R-CNN', fontsize=13)
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'side_by_side_comparison.png'), dpi=150)
plt.show()
print('Saved: results/side_by_side_comparison.png')

## 6. Analysis & Discussion

### Why does one model outperform the other?

**YOLOv8n (single-stage detector)**
- Predicts boxes and classes in a single forward pass → very fast inference
- Uses anchor-free detection with a feature pyramid neck for multi-scale detection
- Pretrained on COCO (80 classes), fine-tuned on our 433-image dataset
- Strength: speed, strong pretrained features, great for real-time use
- Weakness: can struggle with very small or densely packed objects

**Faster R-CNN (two-stage detector)**
- Stage 1: Region Proposal Network (RPN) generates candidate regions
- Stage 2: Classifies and refines each region independently
- More computation per image but typically higher precision on small datasets
- Strength: precise localization, better recall on harder cases
- Weakness: slower, heavier model

### Generalization
With only 433 images, both models rely heavily on ImageNet/COCO pretrained weights.
The diversity of lighting and viewing angles in the dataset is the main challenge.

### Conclusion
For real-time deployment (dashcam, traffic camera) → **YOLOv8** is preferred.  
For highest accuracy in offline analysis → **Faster R-CNN** may be preferred.